# CodeEditor 

`CodeEditor` is a Widget that uses the  Jupyterlab `CodeEditorWrapper` to provide an interface to a [CodeMirror](https://codemirror.net/) editor. 

The code editor provides syntax highlighting according to the specified `mime_type`. 

## Code completion

Code completion is provided for the Python mime types `text/x-python` and `text/x-ipython`. The default invocation for code completion is `Tab`, the same as is used in Jupyterlab.

### Tooltips (Inspect)

Documentation `Tooltips` can be invoked with `Shift Tab`.

## Example

In [ ]:
import anyio
import ipylab
from ipylab.code_editor import CodeEditorOptions

In [ ]:
app = ipylab.JupyterFrontEnd()

In [ ]:
# Default syntax is Python
ce = ipylab.CodeEditor(
    mime_type="text/x-python",
    description="<b>Code editor</b>",
    tooltip="This is a code editor. Code completion is provided for Python",
    value="# Place the cursor in the CodeEditor and press `Shift Enter`\nassert ce.subshell_id is get_ipython().subshell_id\nget_ipython().subshell_id",
    layout={"height": "120px", "overflow": "hidden"},
    description_allow_html=True,
)
# display(ce)
await ce.ready()
# ce.focus()

Invoke the completer with `Tab` and documentation tooltips with `Shift Tab`.

The code in the editor can be evaluated with `Shift Enter`. If there is code selected, it will just evaluate the selected text.

### Kernel Subshell (namespace)

*subject to change*

`CodeEditor` is subclassed from `HasSubshell` and supports evaluation in the user_ns of a shell/subshell specified by the subshell ID. This can be set from another subshell, or a new subshell started by setting the trait `ce.has_subshell = True`.

In [ ]:
ce.has_subshell = True

Now you can go back to the code editor and press `Shift Enter`. Notice there should now be a subshell id printed.

A subshell provides an isolated `user_ns` enabling different objects to have the same name.

In [ ]:
%subshell

## Configuration

A number of editor options are configurable:

In [ ]:
list(CodeEditorOptions.__annotations__)

Overwriting `editor_options` will update the editor (writing to the dict won't).

In [ ]:
ce.editor_options = {
    "autoClosingBrackets": True,
    "matchBrackets": True,
    "highlightTrailingWhitespace": True,
    "highlightWhitespace": True,
}

In [ ]:
values = ["short", "long " * 20, "multi line\n" * 10]


async def test():
    import random

    for _ in range(20):
        ce.value = random.choice(values)
        await anyio.sleep(random.randint(10, 300) / 1e3)

In [ ]:
ce

In [ ]:
await test()

In [ ]:
# Place the label above
ce.layout.flex_flow = "column"

### Add to shell 

Let's add it to the shell.

In [ ]:
# Add the same editor to the shell.
await ce.app.shell.add(ce, mode=ipylab.InsertMode.split_right)

At the moment the context menu will open a console for the main shell. Let's add another context menu item for the code editors subshell.

In [ ]:
selector = ipylab.to_selector(app.vpath, "-editor")
cmd = await app.commands.add_command(
    "Open console for editor", app.shell.open_console, args={"subshell_id": ce.subshell_id}
)
ss = await app.context_menu.add_item(command=cmd, rank=72, args={"subshell_id": ce.subshell_id}, selector=selector)
ce.add_class(selector.removeprefix("."))
ce.focus()

### Other mime_types

Other mime types can be specified. Here we specify markdown.

In [ ]:
md = ipylab.CodeEditor(mime_type="text/x-markdown", value="## Markdown")
md